In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
DATASET_PATH = "../DATA_SET/Crop recommendation dataset.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (57000, 23)
  CROPS TYPE_OF_CROP           SOIL  SEASON SOWN HARVESTED WATER_SOURCE  \
0  rice      cereals  Alluvial soil  kharif  Jun       Sep    irrigated   
1  rice      cereals     Loamy soil  kharif  Jul       Oct      rainfed   
2  rice      cereals      Clay soil  kharif  Jun       Sep    irrigated   
3  rice      cereals  Alluvial soil  kharif  Jul       Oct      rainfed   
4  rice      cereals     Loamy soil  kharif  Jun       Sep    irrigated   

   SOIL_PH  SOIL_PH_HIGH  CROPDURATION  ...  WATERREQUIRED  WATERREQUIRED_MAX  \
0      7.6           8.0         116.9  ...         2462.3               2500   
1      6.2           8.0         117.9  ...         1237.5               2500   
2      6.7           8.0         117.7  ...         1075.1               2500   
3      6.1           8.0         149.8  ...         1549.9               2500   
4      8.0           8.0         131.7  ...         1306.4               2500   

   RELATIVE_HUMIDITY  RELATIVE_HUMI

In [3]:
categorical_columns = [
    "CROPS",
    "TYPE_OF_CROP",
    "SOIL",
    "SEASON",
    "SOWN",
    "HARVESTED",
    "WATER_SOURCE"
]

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("\u00a0", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Standardize soil names
df["SOIL"] = df["SOIL"].str.lower()

print("Dataset cleaning completed.")

Dataset cleaning completed.


In [4]:
FEATURES = [
    "SEASON",
    "WATER_SOURCE",
    "SOIL_PH",
    "TEMP",
    "RELATIVE_HUMIDITY",
    "N",
    "P",
    "K"
]

TARGET = "SOIL"

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("Input features:")
print(FEATURES)

print("\nTarget:", TARGET)

print("\nSamples:", len(X))
print("Soil classes:", y.nunique())

Input features:
['SEASON', 'WATER_SOURCE', 'SOIL_PH', 'TEMP', 'RELATIVE_HUMIDITY', 'N', 'P', 'K']

Target: SOIL

Samples: 57000
Soil classes: 29


In [5]:
print("SOIL CLASSES")
print("=" * 60)

soil_classes = sorted(y.unique())

for i, soil in enumerate(soil_classes):
    print(f"{i:2d} -> {soil}")

print("\nTotal soil classes:", len(soil_classes))

SOIL CLASSES
 0 -> alluvial soil
 1 -> black cotton soil
 2 -> black soil
 3 -> brown loamy soil
 4 -> clay loamy soil
 5 -> clay soil
 6 -> cotton soil
 7 -> deep soil
 8 -> friable soil
 9 -> heavy black soil
10 -> heavy soil
11 -> laterite soil
12 -> light loamy soil
13 -> light soi
14 -> loamy soil
15 -> medium black soil
16 -> red lateritic loamy soil
17 -> red loamy soil
18 -> red soil
19 -> rich red loamy soil
20 -> salty clay loamy soil
21 -> sandy clay loamy soil
22 -> sandy loamy soil
23 -> sandy soil
24 -> shallow black soil
25 -> silty loamy soil
26 -> well-drained loamy soil
27 -> well-drained soil
28 -> well-grained deep loamy moist soil

Total soil classes: 29


In [6]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Encoded soil classes:")
print("=" * 60)

for i, soil in enumerate(label_encoder.classes_):
    print(f"{i:2d} -> {soil}")

Encoded soil classes:
 0 -> alluvial soil
 1 -> black cotton soil
 2 -> black soil
 3 -> brown loamy soil
 4 -> clay loamy soil
 5 -> clay soil
 6 -> cotton soil
 7 -> deep soil
 8 -> friable soil
 9 -> heavy black soil
10 -> heavy soil
11 -> laterite soil
12 -> light loamy soil
13 -> light soi
14 -> loamy soil
15 -> medium black soil
16 -> red lateritic loamy soil
17 -> red loamy soil
18 -> red soil
19 -> rich red loamy soil
20 -> salty clay loamy soil
21 -> sandy clay loamy soil
22 -> sandy loamy soil
23 -> sandy soil
24 -> shallow black soil
25 -> silty loamy soil
26 -> well-drained loamy soil
27 -> well-drained soil
28 -> well-grained deep loamy moist soil


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 45600
Testing samples : 11400


In [8]:
categorical_features = [
    "SEASON",
    "WATER_SOURCE"
]

numerical_features = [
    "SOIL_PH",
    "TEMP",
    "RELATIVE_HUMIDITY",
    "N",
    "P",
    "K"
]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numerical", numerical_pipeline, numerical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape :", X_test_processed.shape)

Training shape: (45600, 11)
Testing shape : (11400, 11)


In [9]:
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [10]:
models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.08,
        max_depth=5,
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    )
}

In [11]:
results = []
trained_models = {}

for name, model in models.items():

    print("\n" + "=" * 70)
    print("TRAINING:", name)
    print("=" * 70)

    model.fit(X_train_processed, y_train)

    y_pred = model.predict(X_test_processed)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")


TRAINING: Random Forest
Accuracy : 0.7389
Precision: 0.8040
Recall   : 0.7389
F1 Score : 0.7521

TRAINING: Extra Trees
Accuracy : 0.7503
Precision: 0.7504
Recall   : 0.7503
F1 Score : 0.7494

TRAINING: Gradient Boosting
Accuracy : 0.7561
Precision: 0.7585
Recall   : 0.7561
F1 Score : 0.7569

TRAINING: Logistic Regression
Accuracy : 0.4893
Precision: 0.4635
Recall   : 0.4893
F1 Score : 0.4673


In [12]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

print("\nSOIL PREDICTION MODEL COMPARISON")
print("=" * 80)

print(results_df.to_string(index=False))


SOIL PREDICTION MODEL COMPARISON
              Model  Accuracy  Precision   Recall  F1 Score
  Gradient Boosting  0.756140   0.758466 0.756140  0.756920
      Random Forest  0.738860   0.803957 0.738860  0.752099
        Extra Trees  0.750263   0.750393 0.750263  0.749367
Logistic Regression  0.489298   0.463493 0.489298  0.467332


In [13]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("\nBEST SOIL PREDICTION MODEL")
print("=" * 60)

print("Model     :", best_model_name)
print("Accuracy  :", results_df.iloc[0]["Accuracy"])
print("Precision :", results_df.iloc[0]["Precision"])
print("Recall    :", results_df.iloc[0]["Recall"])
print("F1 Score  :", results_df.iloc[0]["F1 Score"])


BEST SOIL PREDICTION MODEL
Model     : Gradient Boosting
Accuracy  : 0.756140350877193
Precision : 0.7584664503503598
Recall    : 0.756140350877193
F1 Score  : 0.7569195932864022


In [14]:
best_predictions = best_model.predict(X_test_processed)

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

                                    precision    recall  f1-score   support

                     alluvial soil       0.51      0.57      0.54       333
                 black cotton soil       0.56      0.51      0.54       150
                        black soil       0.19      0.17      0.18       107
                  brown loamy soil       0.17      0.16      0.16        50
                   clay loamy soil       0.49      0.52      0.50       433
                         clay soil       0.72      0.71      0.72       333
                       cotton soil       1.00      1.00      1.00       100
                         deep soil       0.44      0.48      0.46       100
                      friable soil       0.99      0.99      0.99       200
                  heavy black soil       1.00      1.00      1.00       100
                        heavy soil       0.48      0.44      0.46       100
                     laterite soil       0.81      0.84      0.82       700
           

In [16]:
# ============================================================
# SAVE SOIL PREDICTION MODEL - FIXED VERSION
# ============================================================

import os
import joblib
import json

# ------------------------------------------------------------
# 1. Create MODELS folder
# ------------------------------------------------------------

MODELS_DIR = os.path.abspath(
    os.path.join(os.getcwd(), "..", "MODELS")
)

os.makedirs(MODELS_DIR, exist_ok=True)

print("Models folder:")
print(MODELS_DIR)


# ------------------------------------------------------------
# 2. Save BEST MODEL
# ------------------------------------------------------------

joblib.dump(
    best_model,
    os.path.join(
        MODELS_DIR,
        "soil_prediction_model.pkl"
    )
)


# ------------------------------------------------------------
# 3. Save PREPROCESSOR
# ------------------------------------------------------------

joblib.dump(
    preprocessor,
    os.path.join(
        MODELS_DIR,
        "soil_preprocessor.pkl"
    )
)


# ------------------------------------------------------------
# 4. Save LABEL ENCODER
# ------------------------------------------------------------

joblib.dump(
    label_encoder,
    os.path.join(
        MODELS_DIR,
        "soil_label_encoder.pkl"
    )
)


# ------------------------------------------------------------
# 5. Save MODEL COMPARISON
# ------------------------------------------------------------

joblib.dump(
    results,
    os.path.join(
        MODELS_DIR,
        "soil_model_comparison.pkl"
    )
)


# ------------------------------------------------------------
# 6. Create metadata
# ------------------------------------------------------------

metadata = {
    "project": "Smart Farming Soil Prediction Using Machine Learning",

    "task": "Soil Type Prediction",

    "target": "SOIL",

    "features": [
        "SEASON",
        "WATER_SOURCE",
        "SOIL_PH",
        "TEMP",
        "RELATIVE_HUMIDITY",
        "N",
        "P",
        "K"
    ],

    "model": type(best_model).__name__,

    "number_of_classes": len(label_encoder.classes_),

    "soil_classes": label_encoder.classes_.tolist()
}


# ------------------------------------------------------------
# 7. Save metadata
# ------------------------------------------------------------

metadata_path = os.path.join(
    MODELS_DIR,
    "soil_model_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 8. Display saved files
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SOIL MODEL FILES SAVED SUCCESSFULLY")
print("=" * 60)

files_to_check = [
    "soil_prediction_model.pkl",
    "soil_preprocessor.pkl",
    "soil_label_encoder.pkl",
    "soil_model_comparison.pkl",
    "soil_model_metadata.json"
]

for filename in files_to_check:

    file_path = os.path.join(
        MODELS_DIR,
        filename
    )

    if os.path.exists(file_path):

        size_kb = os.path.getsize(file_path) / 1024

        print(
            f"[OK] {filename:<35} "
            f"{size_kb:,.2f} KB"
        )

    else:

        print(
            f"[ERROR] {filename} NOT FOUND"
        )


print("\nLocation:")
print(MODELS_DIR)

Models folder:
c:\Users\Win\Desktop\ITML_15\CODE\MODELS

SOIL MODEL FILES SAVED SUCCESSFULLY
[OK] soil_prediction_model.pkl           19,049.38 KB
[OK] soil_preprocessor.pkl               3.72 KB
[OK] soil_label_encoder.pkl              0.98 KB
[OK] soil_model_comparison.pkl           0.32 KB
[OK] soil_model_metadata.json            1.21 KB

Location:
c:\Users\Win\Desktop\ITML_15\CODE\MODELS
